# Curating the FLEURS Dataset with NeMo Curator

This notebook walks through the FLEURS audio curation pipeline step by step:
1. Download a small FLEURS split
2. Run ASR inference
3. Compute WER and duration
4. Filter by WER threshold
5. Inspect and visualize results

**Requirements**: GPU recommended for ASR inference. Install with `uv sync --extra audio_cuda12`.

In [ ]:
import json
import os
import shutil

from nemo_curator.backends.xenna import XennaExecutor
from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.audio.common import GetAudioDurationStage, PreserveByValueStage
from nemo_curator.stages.audio.datasets.fleurs.create_initial_manifest import CreateInitialManifestFleursStage
from nemo_curator.stages.audio.inference.asr_nemo import InferenceAsrNemoStage
from nemo_curator.stages.audio.io.convert import AudioToDocumentStage
from nemo_curator.stages.audio.metrics.get_wer import GetPairwiseWerStage
from nemo_curator.stages.resources import Resources
from nemo_curator.stages.text.io.writer import JsonlWriter

## Configuration

Adjust these parameters for your setup:

In [ ]:
RAW_DATA_DIR = os.path.abspath("./example_audio/fleurs")
LANG = "en_us"
SPLIT = "dev"
MODEL_NAME = "nvidia/parakeet-tdt-0.6b-v2"
WER_THRESHOLD = 75.0
GPUS = 1.0

RESULT_DIR = os.path.join(RAW_DATA_DIR, "result")
if os.path.isdir(RESULT_DIR):
    shutil.rmtree(RESULT_DIR)

## Step 1: Build the pipeline

The pipeline has 7 stages: download → ASR → WER → duration → filter → convert → write.

In [ ]:
pipeline = Pipeline(
    name="fleurs_tutorial",
    description="Download FLEURS, run ASR, filter by WER",
)

pipeline.add_stage(
    CreateInitialManifestFleursStage(lang=LANG, split=SPLIT, raw_data_dir=RAW_DATA_DIR).with_(batch_size=4)
)
pipeline.add_stage(InferenceAsrNemoStage(model_name=MODEL_NAME).with_(resources=Resources(gpus=GPUS)))
pipeline.add_stage(GetPairwiseWerStage(text_key="text", pred_text_key="pred_text", wer_key="wer"))
pipeline.add_stage(GetAudioDurationStage(audio_filepath_key="audio_filepath", duration_key="duration"))
pipeline.add_stage(PreserveByValueStage(input_value_key="wer", target_value=WER_THRESHOLD, operator="le"))
pipeline.add_stage(AudioToDocumentStage().with_(batch_size=1))
pipeline.add_stage(JsonlWriter(path=RESULT_DIR, write_kwargs={"force_ascii": False}))

print(pipeline.describe())

## Step 2: Execute the pipeline

In [ ]:
executor = XennaExecutor()
pipeline.run(executor)
print("Pipeline complete!")

## Step 3: Load and inspect results

In [ ]:
results = []
for fname in os.listdir(RESULT_DIR):
    if fname.endswith(".jsonl"):
        with open(os.path.join(RESULT_DIR, fname)) as f:
            for line in f:
                if line.strip():
                    results.append(json.loads(line))

print(f"Total samples after filtering: {len(results)}")
print("\nSample entry:")
print(json.dumps(results[0], indent=2, ensure_ascii=False) if results else "No results")

## Step 4: Visualize results

### WER distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

wers = [r.get("wer", 0) for r in results]
durations = [r.get("duration", 0) for r in results]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. WER histogram with threshold line
ax = axes[0, 0]
ax.hist(wers, bins=30, color="#4C72B0", edgecolor="white", alpha=0.85)
ax.axvline(WER_THRESHOLD, color="#C44E52", linestyle="--", linewidth=2, label=f"Threshold ({WER_THRESHOLD}%)")
ax.set_xlabel("WER (%)")
ax.set_ylabel("Count")
ax.set_title("WER Distribution")
ax.legend()

# 2. Duration distribution
ax = axes[0, 1]
ax.hist(durations, bins=30, color="#55A868", edgecolor="white", alpha=0.85)
ax.set_xlabel("Duration (seconds)")
ax.set_ylabel("Count")
ax.set_title("Audio Duration Distribution")

# 3. WER vs Duration scatter
ax = axes[1, 0]
scatter = ax.scatter(durations, wers, c=wers, cmap="RdYlGn_r", alpha=0.6, s=20, edgecolors="none")
ax.axhline(WER_THRESHOLD, color="#C44E52", linestyle="--", linewidth=1.5, alpha=0.7)
ax.set_xlabel("Duration (seconds)")
ax.set_ylabel("WER (%)")
ax.set_title("WER vs Duration")
plt.colorbar(scatter, ax=ax, label="WER %")

# 4. Pass rate at multiple thresholds
ax = axes[1, 1]
thresholds = [5, 10, 25, 50, 75, 100]
pass_rates = [sum(1 for w in wers if w <= t) / len(wers) * 100 for t in thresholds]
bars = ax.bar([str(t) for t in thresholds], pass_rates, color="#8172B2", edgecolor="white")
ax.set_xlabel("WER Threshold (%)")
ax.set_ylabel("Samples Passing (%)")
ax.set_title("Dataset Yield by Threshold")
for bar, rate in zip(bars, pass_rates, strict=True):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{rate:.0f}%", ha="center", fontsize=9)
ax.set_ylim(0, 110)

fig.suptitle(f"FLEURS {LANG} / {SPLIT} — {len(results)} samples (WER ≤ {WER_THRESHOLD}%)", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

print(
    f"\nWER — min: {min(wers):.1f}%, max: {max(wers):.1f}%, mean: {np.mean(wers):.1f}%, median: {np.median(wers):.1f}%"
)
print(f"Duration — min: {min(durations):.2f}s, max: {max(durations):.2f}s, total: {sum(durations):.1f}s")

## Step 5: Experiment with different thresholds

Try changing the WER threshold to see how it affects the dataset size:

In [ ]:
thresholds = [10, 25, 50, 75, 100]
for t in thresholds:
    passing = [r for r in results if r.get("wer", 100) <= t]
    pct = len(passing) / len(results) * 100 if results else 0
    print(f"  WER ≤ {t:3d}%: {len(passing):4d} samples ({pct:.0f}%)")